# Tornado dataset: records, surveys, and county context

Explore the **2010–2025** collection of tornado records, EF ratings, and damage
surveys. This notebook only reads local files. Run
`uv run python download_data.py --verify-downloads` to collect and verify data.
See [README.md](../README.md) and [DATASET.md](../docs/DATASET.md).

| Source | Unit / purpose |
|---|---|
| SPC | Historical tornado tracks and candidate EF labels |
| NCEI Storm Events | County/event details, fatalities, locations, and narratives |
| NWS DAT | Survey points, lines, and polygons; optional damage detail, without photos |
| Census Population Estimates | Annual county population and housing units |
| Census Cartographic Boundaries | Fixed 2020 simplified county map |

These sources count different units; their row totals cannot be added to count
tornadoes. Missing survey records do not mean no tornado occurred. Unknown EF
ratings are retained and must not be converted to EF0.


In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'download_data.py').is_file() and (p / 'pyproject.toml').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from inside the tornado-classification repository.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from dataset_inspection import (load_json, local_path, csv_preview,
                                inventory, reference_check, dat_preview)

DATA = ROOT / 'data'  # Change only if you used download_data.py --data-dir.
START_YEAR, END_YEAR = 2010, 2025
SAMPLE_ROWS = 10
NCEI_YEAR = 2025
pd.set_option('display.max_columns', 16)
pd.set_option('display.max_colwidth', 80)
print(f'Inspecting {DATA}; period {START_YEAR}–{END_YEAR}')
print('All cells are read-only. No downloads are triggered.')


## Download status and source inventory

The NOAA download manifest covers SPC, NCEI, and DAT; a separate Census manifest
covers the two context sources. Check the verification scope as well as its status. The independent verification
report checks saved files, filtering, and survey batches. Successful retrieval
does not establish complete historical survey coverage. Inventory includes caches;
use the manifest and DAT indexes to identify the active selection.


In [ ]:
manifest = load_json(DATA / f'download_manifest_{START_YEAR}_{END_YEAR}.json', {})
quality = load_json(DATA / f'quality_summary_{START_YEAR}_{END_YEAR}.json', {})
verification = load_json(DATA / f'download_verification_{START_YEAR}_{END_YEAR}.json', {})
census_manifest = load_json(DATA / f'census_manifest_{START_YEAR}_{END_YEAR}.json', {})
status = [
    {'scope': 'SPC / NCEI / DAT',
     'status': 'available' if manifest.get('requested_archives_available') else 'incomplete' if manifest else 'not downloaded'},
    {'scope': 'Census estimates / county map', 'status': census_manifest.get('status', 'not downloaded')},
    {'scope': 'Verification: ' + verification.get('scope', 'unknown'), 'status': verification.get('status', 'not run')},
]
display(pd.DataFrame(status))
if verification.get('manifest'):
    display(pd.DataFrame([reference_check(DATA, verification['manifest'])]))
if verification.get('census_manifest'):
    display(pd.DataFrame([reference_check(DATA, verification['census_manifest'])]))
files = pd.DataFrame(inventory(DATA))
if not files.empty:
    display(files)
else:
    print('No dataset files yet. Run the download script separately when ready.')


## SPC tracks and EF-label balance

SPC is the primary event catalog. These are historical single-track records, not preliminary daily reports. The default period is entirely within the EF era (which began February 1, 2007). The choice of 2010 is project scope, not a rating-scale transition or a DAT-completeness cutoff.


In [ ]:
spc_output = next((row for row in manifest.get('outputs', []) if row['source'] == 'SPC'), None)
spc = pd.DataFrame()
if spc_output:
    spc_path = local_path(DATA, spc_output['path'])
    if spc_path.is_file():
        spc = pd.read_csv(spc_path, dtype=str, keep_default_na=False)
if not spc.empty:
    print(f'{len(spc):,} SPC track records')
    columns = [c for c in ['yr','om','date','time','tz','st','mag','slat','slon','len','wid'] if c in spc]
    display(spc[columns].head(SAMPLE_ROWS))
    ratings = spc['mag'].where(spc['mag'].isin(list('012345')), 'Unknown')
    rating_counts = ratings.value_counts().reindex([*list('012345'), 'Unknown'], fill_value=0)
    display(rating_counts.rename_axis('EF rating').to_frame('tracks'))
    ax = rating_counts.plot.bar(figsize=(8, 3), color='#2878a8', title='SPC recorded EF ratings')
    ax.set_ylabel('Tracks'); ax.set_xlabel('EF rating'); ax.tick_params(axis='x', rotation=0)
    plt.tight_layout(); plt.show()
else:
    print('SPC data not available in the selected manifest yet.')


## Original-source yearly coverage and survey quality

Annual outputs with zero records differ from unavailable outputs. DAT quality counts describe survey features, not unique tornadoes. A DAT point rating need not equal the maximum rating of its tornado. The downloaded summary is a saved report; this notebook does not regenerate it.


In [ ]:
coverage_rows = []
for item in manifest.get('year_coverage', []):
    for year in range(START_YEAR, END_YEAR + 1):
        coverage_rows.append({'source': item['source'], 'product': item['product'], 'year': year,
                              'rows': item['rows_by_year'].get(str(year), 0),
                              'available': year in item['available_years']})
if coverage_rows:
    display(pd.DataFrame(coverage_rows).pivot(index='year', columns=['source','product'], values='rows'))
    unavailable = pd.DataFrame(coverage_rows).query('not available')
    if not unavailable.empty:
        display(unavailable)
else:
    print('No original-source coverage report yet.')
if quality:
    quality_ref = {'path': quality['download_manifest'], 'sha256': quality['download_manifest_sha256']}
    display(pd.DataFrame([reference_check(DATA, quality_ref)]))
    dat_quality = [{'year': item['year'], 'layer': item['layer'], **item['counts']}
                   for item in quality.get('dat_by_year_layer', [])]
    if dat_quality:
        display(pd.DataFrame(dat_quality).groupby('layer').sum(numeric_only=True).drop(columns='year', errors='ignore'))
else:
    print('No saved label/survey-quality summary yet.')


## NCEI and DAT record samples

NCEI's raw annual archives include all hazards; these samples use tornado-only extracts. CSV previews retain codes as strings and only read a few rows. DAT samples read one batch named in an index. Neither preview performs event matching.


In [ ]:
for table in ('details', 'fatalities', 'locations'):
    output = next((row for row in manifest.get('outputs', [])
                   if row.get('source') == 'NCEI' and row.get('year') == NCEI_YEAR and row.get('table') == table), None)
    rows = csv_preview(local_path(DATA, output['path']), SAMPLE_ROWS) if output else []
    print(f'NCEI {NCEI_YEAR} {table}:')
    display(pd.DataFrame(rows)) if rows else print('No local sample available.')
for layer in ('points', 'lines', 'polygons'):
    rows = dat_preview(DATA, manifest, layer, SAMPLE_ROWS)
    print(f'DAT {layer}:')
    if rows:
        frame = pd.DataFrame(rows)
        useful = [c for c in ['objectid','event_id','stormdate','efscale','di','dod','windspeed','geometry_type'] if c in frame]
        display(frame[useful] if useful else frame)
    else:
        print('No indexed local sample available.')


## County population and housing context

One row per county/year, covering the 50 states and DC. These are July 1 estimates:
vintage 2020 supplies 2010–2019, and vintage 2025 supplies 2020–2025. They use each
release's county definitions, not necessarily those in effect on the tornado date.

Housing units are residences, not building counts. This table is not joined to
tornadoes or paths. A matching map FIPS code does not establish compatible boundaries;
Connecticut's newer planning regions are flagged as absent from the fixed 2020 map.
Do not fill missing joins with zero or interpret county totals as people directly hit.


In [ ]:
county_context = pd.DataFrame()
if census_manifest.get('status') == 'complete':
    context_path = local_path(DATA, census_manifest['context']['path'])
    if context_path.is_file():
        county_context = pd.read_csv(context_path, dtype={'county_fips': str, 'map_2020_fips_present': str})
if not county_context.empty:
    display(county_context.head(SAMPLE_ROWS))
    display(county_context.groupby(['year', 'estimate_vintage']).agg(
        counties=('county_fips', 'size'), population=('population', 'sum'),
        housing_units=('housing_units', 'sum')))
    unmatched = county_context.loc[county_context['map_2020_fips_present'].eq('false'),
                                   ['county_fips', 'state_name', 'county_name']].drop_duplicates()
    print(f'{len(unmatched)} county codes absent from the fixed 2020 map:')
    if not unmatched.empty:
        display(unmatched)
else:
    print('No completed Census context collection yet.')


## Simplified county map and tornado locations

The 2020 Census cartographic map is generalized at 1:5,000,000 and converted from
WGS84 KML to GeoJSON, preserving polygon parts and holes. It includes Puerto Rico and other US territories;
population/housing tables cover the 50 states and DC. The preview below shows the
contiguous US and SPC start points. It performs no spatial joins or exposure estimates.
Use DAT survey lines/polygons for available damage paths; connecting SPC endpoints
only approximates a track. Map boundaries are for display, not precise path intersections.


In [ ]:
from matplotlib.collections import LineCollection

county_map = {}
if census_manifest.get('status') == 'complete':
    county_map = load_json(local_path(DATA, census_manifest['boundaries']['path']), {})
if county_map.get('features'):
    rings = [ring for feature in county_map['features']
             for polygon in feature['geometry']['coordinates'] for ring in polygon]
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.add_collection(LineCollection(rings, colors='#aab3ba', linewidths=0.25))
    if not spc.empty and {'slon', 'slat'}.issubset(spc.columns):
        ax.scatter(pd.to_numeric(spc['slon'], errors='coerce'),
                   pd.to_numeric(spc['slat'], errors='coerce'),
                   s=2, alpha=0.18, color='#b84235', rasterized=True, label='SPC start locations')
        ax.legend(loc='lower left')
    ax.set(xlim=(-125, -66), ylim=(24, 50), xlabel='Longitude', ylabel='Latitude',
           title='Tornado start locations and simplified 2020 counties — contiguous US')
    ax.set_aspect(1.3)
    plt.tight_layout(); plt.show()
else:
    print('No completed county map collection yet.')


## Interpretation before analysis

- Use SPC as the 2010–2025 master event catalog; DAT availability should not define inclusion.
- Reconcile event identities and labels before combining survey records.
- Preserve unknown ratings; exclude them only when an experiment needs known supervised labels.
- Treat Census counts as county context; reconcile geographic vintages before event joins.
- Explore EF-label distributions, regional/yearly patterns, source agreement, and survey coverage.
- Keep EF-revealing narratives, survey wind estimates, and rating-derived fields out of classifier predictors.
- Group related records before train/test splits and assess rare EF classes carefully.

The [DAT coverage report](../reports/dat_coverage/dat_coverage_report.md) describes its stated
snapshot. These inspections provide a starting point; event matching and
model-feature preparation remain separate work.
